<a href="https://colab.research.google.com/github/ywchanna2001/LLM-Finetunning/blob/main/Guanaco_Llama_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guanaco → Llama 2 dataset (1k samples)

Reformats `timdettmers/openassistant-guanaco` into the Llama 2 chat template and pushes it to your Hugging Face account.

**Before you start:** create a Hugging Face access token with **Write** permission at https://huggingface.co/settings/tokens

## Step 1: Install and log in

In [ ]:
!pip install -q -U datasets huggingface_hub

In [ ]:
# `!huggingface-cli login` no longer works, and `!hf auth login` can't prompt for input
# from a Colab cell. Use the notebook login widget instead and paste your WRITE token.
#
# Tip: you can instead add the token under Colab's "Secrets" (key icon on the left) with
# the name HF_TOKEN and turn on notebook access; then this cell logs in without prompting.
from huggingface_hub import login, notebook_login, whoami

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    notebook_login()

In [ ]:
# Check that the login worked and see which account you're pushing to
user = whoami()
hf_username = user["name"]
print("Logged in as:", hf_username)

## Step 2: Load and reformat the dataset

In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("timdettmers/openassistant-guanaco")

# Shuffle the dataset and keep 1,000 examples
dataset = dataset["train"].shuffle(seed=42).select(range(1000))

# Convert "### Human: ... ### Assistant: ..." into the Llama 2 template
def transform_conversation(example):
    conversation_text = example["text"]
    segments = conversation_text.split("###")

    reformatted_segments = []

    # Iterate over (Human, Assistant) pairs
    for i in range(1, len(segments) - 1, 2):
        human_text = segments[i].strip().replace("Human:", "").strip()

        if i + 1 < len(segments):
            assistant_text = segments[i + 1].strip().replace("Assistant:", "").strip()
            reformatted_segments.append(f"<s>[INST] {human_text} [/INST] {assistant_text} </s>")
        else:
            reformatted_segments.append(f"<s>[INST] {human_text} [/INST] </s>")

    return {"text": "".join(reformatted_segments)}

# Apply the transformation
transformed_dataset = dataset.map(transform_conversation)

In [ ]:
# Look at one example to check the format
print(transformed_dataset[0]["text"][:500])

## Step 3: Push to the Hugging Face Hub

In [ ]:
# The repo name must include your username, e.g. "your-name/guanaco-llama2-1k".
# Without it the Hub can't tell where to create the dataset.
repo_id = f"{hf_username}/guanaco-llama2-1k"

transformed_dataset.push_to_hub(repo_id)
print(f"Done: https://huggingface.co/datasets/{repo_id}")

To fine-tune on your own copy, set this in the fine-tuning notebook:

```python
dataset_name = "<your-username>/guanaco-llama2-1k"
```